Update 231210: 
Site annotations corrected using hg38 version information (previously used hg37 version): modified in GEMMA_Output/add_loc

Update 231117: 
Request the consolidated ranking of each selected ageCpG across various models: see "ageCpGs' span-model ranks"

## Set up Environment

- Suitable for the integrated analysis of multiple "*.assoc.txt" files output by GEMMA.
- Place all "*.assoc.txt" files to be analyzed in the "./data/" directory of this IPython notebook; this directory must not contain any files other than those to be analyzed.

In [8]:
PATH_data = input('Please input the path of datas. Default: ./data/')
PATH_probe_doc = input('Please input the path of illumina product file.')#'/extData/Ken/Documents/infinium-methylationepic-v-1-0-b5-manifest-file_for_pandas.csv'

import pandas as pd
import matplotlib.pyplot as plt
import os
import re

Please input the path of datas. Default: "./data/" ./lmm26to29/
Please input the path of documentations. /extData/Ken/Documents/infinium-methylationepic-v-1-0-b5-manifest-file_for_pandas.csv


In [3]:
class GEMMA_Output:
    
    def __init__(self,note='None'):
        self.note = note
    
    def load_data(self,path,file_name):
        self.name = file_name.split('_')[0]
        df = pd.read_csv(path+file_name,sep='\t')
        self.data_df = df
        ranked_df = df.sort_values(by='p_wald')
        ranked_df.reset_index(inplace=True,drop=True)
        ranked_df['rank']=ranked_df.index
        ranked_df.set_index('rs',inplace=True)
        self.rank_df = ranked_df[['rank','p_wald']]
    
    def add_loc(self,doc_df):
        merge_df = pd.merge(doc_df[['Genome_Build','CHR_hg38','Start_hg38','SourceSeq','UCSC_RefGene_Name','UCSC_RefGene_Accession', 'Relation_to_UCSC_CpG_Island']],self.rank_df[['rank','p_wald']],how='inner',left_index=True,right_index=True)
        merge_df['single_UCSC_name'] = merge_df['UCSC_RefGene_Name'].str.replace('(;.*)','',regex=True)
        self.rank_df = merge_df.sort_values(by='p_wald')
    
    def select_sites(self,alpha=0.05):
        self.alpha = alpha
        self.threshold_p = alpha/len(self.rank_df.index)
        self.select_df = self.rank_df[self.rank_df['p_wald']<self.threshold_p]
    
    def show_info(self):
        print('Name:',self.name)
        print('Note:',self.note)
        print('alpha:',self.alpha)
        print('threshold_p (alpha/counts(sites)):',self.threshold_p)
        print('selected_sites_counts:',len(self.select_df.index))
        print('selected_sites:\n',self.select_df.head,sep='')
        print('------------------------------')

## Load Data

**Load Documentation**  
Load the probe annotation document（illumina product file）

In [4]:
doc_df = pd.read_csv(PATH_probe_doc,low_memory=False)
doc_df.set_index('Name',inplace=True)
print(doc_df.columns)
doc_df

Index(['IlmnID', 'AddressA_ID', 'AlleleA_ProbeSeq', 'AddressB_ID',
       'AlleleB_ProbeSeq', 'Infinium_Design_Type', 'Next_Base',
       'Color_Channel', 'Forward_Sequence', 'Genome_Build', 'CHR', 'MAPINFO',
       'SourceSeq', 'Strand', 'UCSC_RefGene_Name', 'UCSC_RefGene_Accession',
       'UCSC_RefGene_Group', 'UCSC_CpG_Islands_Name',
       'Relation_to_UCSC_CpG_Island', 'Phantom4_Enhancers',
       'Phantom5_Enhancers', 'DMR', '450k_Enhancer', 'HMM_Island',
       'Regulatory_Feature_Name', 'Regulatory_Feature_Group',
       'GencodeBasicV12_NAME', 'GencodeBasicV12_Accession',
       'GencodeBasicV12_Group', 'GencodeCompV12_NAME',
       'GencodeCompV12_Accession', 'GencodeCompV12_Group',
       'DNase_Hypersensitivity_NAME', 'DNase_Hypersensitivity_Evidence_Count',
       'OpenChromatin_NAME', 'OpenChromatin_Evidence_Count', 'TFBS_NAME',
       'TFBS_Evidence_Count', 'Methyl27_Loci', 'Methyl450_Loci',
       'Chromosome_36', 'Coordinate_36', 'SNP_ID', 'SNP_DISTANCE',
       'SNP_

,IlmnID,AddressA_ID,AlleleA_ProbeSeq,AddressB_ID,AlleleB_ProbeSeq,Infinium_Design_Type,Next_Base,Color_Channel,Forward_Sequence,Genome_Build,...,Coordinate_36,SNP_ID,SNP_DISTANCE,SNP_MinorAlleleFrequency,Random_Loci,MFG_Change_Flagged,CHR_hg38,Start_hg38,End_hg38,Strand_hg38
Name,,,,,,,,,,,,,,,,,,,,,
cg07881041,cg07881041,85713262,CTACAAATACAACACCCTCAACCCATATTTCATATATTATCTCATT...,NaN,NaN,II,NaN,NaN,CTGCACGCCTACTGCAGGTGCAGCACCCTCAGCCCATGTTTCATGT...,37.0,...,5187016,rs187313142,18,0.0002,NaN,False,chr19,5236004.0,5236006.0,+
cg18478105,cg18478105,46761277,AAATAAATTTCACTCTCAAATCCCAATCTCATACAACAAAACAAAA...,86644198.0,AAATAAATTTCGCTCTCAAATCCCAATCTCGTACGACGAAACGAAA...,I,C,Grn,TCCCGTCTTACGGGATGGATTTCGCTCTCAGGTCCCAGTCTCGTGC...,37.0,...,61318095,rs549944121,5,0.001797,NaN,False,chr20,63216297.0,63216299.0,+
cg23229610,cg23229610,21717843,ATAAAATTCTTTCCTTAAAAAACATTAAAACCAAAATAAACAAAAA...,NaN,NaN,II,NaN,NaN,GTTTCTGGACAGTAAAATTCTTTCCTTGAAGGACATTAGGGCCAAA...,37.0,...,6763712,rs545824288;rs527255711,40;12,0.000200;0.001198,NaN,False,chr1,6781064.0,6781066.0,+
cg03513874,cg03513874,29622133,ACAATAAAATAATAAAATCCCATCACTACTTACCCTCCTTAAATAA...,NaN,NaN,II,NaN,NaN,ATTGTGCCCACCTTGCTGCTGACAGTTAAGCATCACTAAAGTAGGA...,37.0,...,198011711,NaN,NaN,NaN,NaN,False,chr2,197438741.0,197438743.0,-
cg09835024,cg09835024,16745152,AATAAACACCAACCCCAAACCAATCTCACTTTATTAAATTACAAAA...,81631976.0,AATAAACGCCGACCCCGAACCGATCTCGCTTTATTAAATTACAAAA...,I,A,Red,AGCCCCGTCATAGGTGGGCGCCGACCCCGAGCCGATCTCGCTTTAT...,37.0,...,23982561,NaN,NaN,NaN,NaN,False,chrX,24054522.0,24054524.0,+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
cg23079522,cg23079522,98681180,AAAAAAATACCCAAAACACATTCTACTATTACTCCRTAATAACCCT...,NaN,NaN,II,NaN,NaN,AGGAGGAAGGTGGAAAGGTACCCAAGACACATTCTGCTGTTACTCC...,37.0,...,162052322,rs144073549;rs556478458,40;2,0.003594;0.000399,NaN,False,chr3,160851839.0,160851841.0,+
cg16818145,cg16818145,67613963,CAAAAAACRCTTAATAAACACCCAAAAAACTCCRTAAAAACTTCTA...,NaN,NaN,II,NaN,NaN,TTCCCTCGCTCCAGGAGACGCTTGGTGGGCACCCAGGAAGCTCCGT...,37.0,...,184264971,rs536633857;rs548317428,18;1,0.000200;0.000998,NaN,False,chr3,183064488.0,183064490.0,+
cg14585103,cg14585103,71743112,ATATCTAAATTTTCCAAAAACAATCCTAACTACATAACATAACATT...,NaN,NaN,II,NaN,NaN,GAGAAATAATAATGTCTGAATTTTCCAGAGACAGTCCTGGCTACAT...,37.0,...,140009790,rs562900783;rs528956739;rs11780350;rs147800551,48;46;42;18,0.000200;0.000200;0.033147;0.004193,NaN,False,chr8,138928364.0,138928366.0,+


**Check the file list of datas to be analyzed**  

In [5]:
LIST_files = os.listdir(PATH_data)
LIST_files

['LMM26_lmm.assoc.txt',
 'LMM27_lmm.assoc.txt',
 'LMM28_lmm.assoc.txt',
 'LMM29_lmm.assoc.txt',
 'LMM20_lmm.assoc.txt']

In [6]:
LIST_files.sort()
LIST_files

['LMM20_lmm.assoc.txt',
 'LMM26_lmm.assoc.txt',
 'LMM27_lmm.assoc.txt',
 'LMM28_lmm.assoc.txt',
 'LMM29_lmm.assoc.txt']

**Load Data**  
When reading data, the filename of each DEWAS result (for example, aabbcc.assoc.txt) (aabbcc) will automatically be available as the class instance name for invocation.

In [9]:
LIST_data = []

for i in range(len(LIST_files)):
    file_name = LIST_files[i]
    dataname = file_name.split('.')[0]
    print(i+1,'/',len(LIST_files),'Loading',file_name,'...')
    globals()[dataname] = GEMMA_Output()
    exec(dataname+'.load_data(PATH_data,file_name)')
    LIST_data.append(dataname)

print('\n','Data are loaded and callable with:',LIST_data,sep='\n')

1 / 5 Loading LMM20_lmm.assoc.txt ...
2 / 5 Loading LMM26_lmm.assoc.txt ...
3 / 5 Loading LMM27_lmm.assoc.txt ...
4 / 5 Loading LMM28_lmm.assoc.txt ...
5 / 5 Loading LMM29_lmm.assoc.txt ...


Data are loaded and callable with:
['LMM20', 'LMM26', 'LMM27', 'LMM28', 'LMM29']


**Add location info**  
Add the biological location and information of associated genes.

In [10]:
for i in range(len(LIST_data)):
    print(i+1,'/',len(LIST_data),'Processing loc info of',LIST_data[i],'...')
    exec(LIST_data[i]+'.add_loc(doc_df)')

1 / 5 Processing loc info of LMM20 ...
2 / 5 Processing loc info of LMM26 ...
3 / 5 Processing loc info of LMM27 ...
4 / 5 Processing loc info of LMM28 ...
5 / 5 Processing loc info of LMM29 ...


**Add/Modify Notes of datas**  

In [11]:
for i in range(len(LIST_data)):
    print(i+1,'/',len(LIST_data),'Please input the note of',LIST_data[i],':')
    note = input()
    exec(LIST_data[i]+'.note = note')

1 / 5 Please input the note of LMM20 :


 continuous


2 / 5 Please input the note of LMM26 :


 D1090


3 / 5 Please input the note of LMM27 :


 D1585


4 / 5 Please input the note of LMM28 :


 D2080


5 / 5 Please input the note of LMM29 :


 D2575


## Significant Sites Selection

**Note**
- The threshold of p_value equals: alpha/counts(probes)  
- Alpha could be 0.5, 0.05 and so on. **Recomended and default value is 0.05**

In [12]:
alpha = float(input('Please input the alpha, 0.05 is recommended:')) #0.05

Please input the alpha, 0.05 is recommended: 0.05


In [13]:
for i in range(len(LIST_data)):
    exec(LIST_data[i]+'.select_sites(alpha)')
    exec(LIST_data[i]+'.show_info()')
    #print(i+1,'/',len(LIST_data),'The alpha of',LIST_data[i],'has been set to',alpha)

Name: LMM20
Note: continuous
alpha: 0.05
threshold_p (alpha/counts(sites)): 7.470804097586632e-08
selected_sites_counts: 599
selected_sites:
<bound method NDFrame.head of             Genome_Build CHR_hg38   Start_hg38  \
cg16867657          37.0     chr6   11044643.0   
cg08097417          37.0     chr7  130734371.0   
cg06639320          37.0     chr2  105399281.0   
cg17268658          37.0     chr2  105399287.0   
cg23606718          37.0     chr2  130756353.0   
...                  ...      ...          ...   
cg10757709          37.0     chr6     491346.0   
cg23756236          37.0    chr19   37334669.0   
cg04455430          37.0     chr1   50421247.0   
cg22719623          37.0     chr6  154039596.0   
cg10002915          37.0     chr4  148448022.0   

                                                    SourceSeq  \
cg16867657  CGGCGGCTCAACGTCCACGGAGCCCCAGGAATACCCACCCGCTGCC...   
cg08097417  TGTTTAACAGCCTCAGAAATTATCTTGTCTCCGCGTTCTTTCTTCT...   
cg06639320  AGGGCTCCTTTCTTCGTGCCC

---

## Comparison of Significant Loci Rankings Across Models

In [17]:
uni_select_df = pd.DataFrame()

for i in range(len(LIST_data)):
    exec('temp_df='+LIST_data[i]+".select_df[['p_wald','rank']]")
    uni_select_df = pd.merge(uni_select_df,temp_df,left_index=True,right_index=True,how='outer')
    uni_select_df.rename(columns={'p_wald':LIST_data[i]+'_p_wald','rank':LIST_data[i]+'_rank'},inplace=True)

uni_select_df['!UCSC_gene_name']=doc_df.loc[uni_select_df.index]['UCSC_RefGene_Name'].str.replace('(;.*)','',regex=True)

In [78]:
uni_rank_df = uni_select_df[['!UCSC_gene_name','LMM20_rank','LMM26_rank','LMM27_rank','LMM28_rank','LMM29_rank']].sort_values(by='LMM20_rank')
uni_rank_df[:20]

,!UCSC_gene_name,LMM20_rank,LMM26_rank,LMM27_rank,LMM28_rank,LMM29_rank
cg16867657,ELOVL2,0.0,105.0,112.0,166.0,12.0
cg08097417,KLF14,1.0,2.0,3.0,10.0,NaN
cg06639320,FHL2,2.0,NaN,NaN,NaN,11.0
cg17268658,FHL2,3.0,NaN,NaN,NaN,24.0
cg23606718,FAM123C,4.0,22.0,1.0,8.0,NaN
cg21572722,ELOVL2,5.0,NaN,NaN,3.0,NaN
cg17110586,NaN,6.0,NaN,26.0,80.0,5.0
cg11071401,CACNA1G,7.0,NaN,NaN,NaN,NaN
cg07553761,TRIM59,8.0,NaN,NaN,NaN,NaN
cg22736354,NHLRC1,9.0,NaN,266.0,17.0,4.0


In [ ]:
uni_rank_df.to_csv('./ageCpGs.csv')